In [ ]:
import math

# Вспомогательные функции
def factorial(n):
    return math.factorial(n)

def mask_to_coalition(mask, n):
    """Преобразует битовую маску (целое число) в множество игроков (с номерами от 1 до n)."""
    return {i+1 for i in range(n) if mask & (1 << i)}

# 1. Проверка супераддитивности с подробным выводом неравенств и исправлением ХФ
def check_and_fix_superadditivity(v, n):
    """
    Для всех двух непересекающихся коалиций S и T (S ≠ ∅, T ≠ ∅)
    проверяется неравенство:
      v(S ∪ T) ≥ v(S) + v(T).
    Для каждой такой пары выводится само неравенство.
    Если неравенство нарушается, выводится сообщение и 
    характеристическая функция исправляется: v(S ∪ T) := v(S) + v(T).
    """
    fixed = False
    for S in range(2**n):
        for T in range(2**n):
            # Рассматриваем только непустые и непересекающиеся коалиции
            if S & T == 0 and S != 0 and T != 0:
                union = S | T
                left_side = v[union]
                right_side = v[S] + v[T]
                inequality = (f"v({mask_to_coalition(union, n)}) = {left_side}   ≥   "
                              f"v({mask_to_coalition(S, n)}) + v({mask_to_coalition(T, n)}) = {v[S]} + {v[T]} = {right_side}")
                # Вывод неравенства
                if left_side < right_side:
                    print("Нарушение супераддиётивности:")
                    print("  " + inequality)
                    print("  => Условие НЕ выполнено. Исправляем: v(S∪T) := v(S) + v(T).\n")
                    v[union] = right_side
                    fixed = True
                else:
                    print("Проверка супераддитивности:")
                    print("  " + inequality)
                    print("  => Условие выполнено.\n")
    return fixed

# 2. Проверка выпуклости (выводим только нарушения)
def check_convexity(v, n):
    """
    Проверяет условие выпуклости: для любых S, T должно выполняться:
      v(S ∪ T) + v(S ∩ T) ≥ v(S) + v(T)
    При нарушении выводит подробное сообщение.
    """
    convex = True
    for S in range(2**n):
        for T in range(2**n):
            union = S | T           # объединение
            intersection = S & T    # пересечение
            if v[union] + v[intersection] < v[S] + v[T]:
                print("Нарушена выпуклость:")
                print(f"  S = {mask_to_coalition(S, n)}, T = {mask_to_coalition(T, n)}")
                print(f"  S ∪ T = {mask_to_coalition(union, n)} имеет v = {v[union]}")
                print(f"  S ∩ T = {mask_to_coalition(intersection, n)} имеет v = {v[intersection]}")
                print(f"  v(S) + v(T) = {v[S]} + {v[T]} = {v[S] + v[T]}")
                print(f"  v(S ∪ T) + v(S ∩ T) = {v[union]} + {v[intersection]} = {v[union] + v[intersection]}\n")
                convex = False
    return convex

# 3. Вычисление вектора Шепли по формуле:
#    x_i(v)=1/N! * sum_{S: i in S} (|S|-1)! (N-|S|)! (v(S)-v(S\{i}))
def shapley_value(v, n):
    phi = [0.0] * n
    N_fact = factorial(n)
    for i in range(n):
        bit_i = 1 << i  # бит, соответствующий игроку i (игроки нумеруются от 0 до n-1)
        for S in range(2**n):
            # Суммируем по всем S, содержащим i
            if S & bit_i != 0:
                s_size = bin(S).count("1")
                S_minus_i = S & ~bit_i
                coef = factorial(s_size - 1) * factorial(n - s_size) / N_fact
                phi[i] += coef * (v[S] - v[S_minus_i])
    return phi



In [ ]:
# Задаём характеристическую функцию для n = 4 игроков.
# Ключи словаря — битовые маски от 0 до 15, где бит i соответствует игроку i+1.
v = {
    0: 0,      # 0
    1: 4,      # {1}    0001
    2: 1,      # {2}    0010
    4: 1,      # {3}    0100
    8: 1,      # {4}    1000
    3: 7,      # {1,2}  0011
    5: 7,      # {1,3}  0101
    9: 7,      # {1,4}  ...
    6: 3,      # {2,3}
    10: 2,     # {2,4}
    12: 2,     # {3,4} 
    7: 10,     # {1,2,3}
    11: 10,    # {1,2,4}
    13: 10,    # {1,3,4}
    14: 6,     # {2,3,4}
    15: 12     # {1,2,3,4}
}

print("Проверка супераддитивности:")
if check_and_fix_superadditivity(v, 4):
    print("Характеристическая функция была исправлена для достижения супераддитивности.\n")
else:
    print("Игра уже супераддитивна.\n")

print("Проверка выпуклости:")
if check_convexity(v, 4):
    print("Игра выпуклая (convex).\n")
else:
    print("Игра НЕ выпуклая (not convex).\n")

# 4. Вычисление вектора Шепли
phi = shapley_value(v, 4)
print("Вектор Шепли:")
for i, val in enumerate(phi):
    print(f"  Игрок {i+1}: φ = {val:.4f}")
print(f"Сумма компонент вектора Шепли: {sum(phi):.4f} (должна равняться v({mask_to_coalition(15, 4)}) = {v[15]})\n")

# 5. Проверка рационализации:
# 5.1 Проверка индивидуальной рационализации
print("Проверка индивидуальной рационализации:")
individual_ok = True
for i in range(4):
    single_mask = 1 << i
    # Вывод строки вида: x{1} = <полученная величина> >= u{1} = <заданная>
    print(f"x{{{i+1}}} = {phi[i]:.4f} >= u{{{i+1}}} = {v[single_mask]}")
    if phi[i] < v[single_mask]:
        individual_ok = False
if individual_ok:
    print("Индивидуальная рационализация выполняется для всех игроков.\n"
          "Каждый игрок получает не менее своей индивидуальной ценности, заданной характеристической функцией.\n")
else:
    print("Индивидуальная рационализация НЕ выполняется.\n")

# 5.2 Проверка групповой рационализации
print("Проверка групповой рационализации:")
total_phi = sum(phi)
print(f"Сумма x{{i}} = {total_phi:.4f} >= u{{1,2,3,4}} = {v[15]}")
if abs(total_phi - v[15]) < 1e-6:
    print("Групповая рационализация выполняется: общая распределённая выгода равна общей ценности коалиции всех игроков.\n")
else:
    print("Групповая рационализация НЕ выполняется.\n")